# 🔬 Demo End-to-End: XAI + AI Agent Pipeline
## Multimodal Restaurant Review Scoring — CrossAttentionFusion (PhoBERT × Swin-B)

**Nhóm 24 — SE365 | Trình bày: Demo cuối kỳ**

Pipeline đầy đủ:
1. **Dự đoán** (CrossAttentionFusion) → 5 điểm: Food / Price / Atmosphere / Service / Overall
2. **Grad-CAM** — vùng ảnh quan trọng
3. **PhoBERT Attention** — từ ngữ nổi bật trong review
4. **Cross-Attention** — tương tác text ↔ image
5. **SHAP** — đóng góp từng chiều fused embedding [1024]
6. **LIME** — giải thích cục bộ (text + image)
7. **AI Agent** (GPT-4o) — báo cáo tổng hợp tiếng Việt

> **3 mẫu thử:** Mẫu A (dự đoán chính xác), Mẫu B (lỗi/xung đột), Mẫu C (đa ảnh phong phú)

---
## 0.2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')

---
## 0.3 · Clone repo & checkout branch `final_demo`

In [ ]:
import os

REPO_URL = 'https://github.com/luubinhdeptrai/SE365.git'  # adjust if private
REPO_DIR = '/content/SE365'
BRANCH   = 'final_demo'

if not os.path.isdir(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
os.system(f'git fetch origin {BRANCH}')
os.system(f'git checkout {BRANCH}')
os.system(f'git pull origin {BRANCH}')

print(f'✅ Working directory: {os.getcwd()}')
print('✅ Branch: ', end='')
os.system('git branch --show-current')

---
## 0.4 · Install thư viện bổ sung (nếu thiếu)

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg, name in [('shap','shap'),('lime','lime'),('timm','timm'),
                   ('openai','openai'),('seaborn','seaborn'),('scikit-image','skimage')]:
    try:
        __import__(name)
    except ImportError:
        _pip(pkg)

print('✅ Tất cả thư viện đã sẵn sàng.')

---
## 0.5 · Imports

In [ ]:
import os, sys, json, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import torch

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
warnings.filterwarnings('ignore')

REPO_DIR = '/content/SE365'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from xai.config import (
    TARGET_NAMES, DISPLAY_NAMES, FACTOR_NAMES,
    FUSED_DIM, CROSS_ATTN_HIDDEN_DIM,
    COLOR_SCHEMES, DEFAULT_DPI, THESIS_DPI,
    BEST_EXP_ID, DEFAULT_SEED,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    load_model, get_tokenizer, get_image_processor,
    load_single_sample, get_prediction,
    enable_eager_attention, get_device, set_seed,
)
from xai.gradcam_explainer import (
    compute_gradcam_for_image, overlay_cam_on_image, find_target_layer,
)
from xai.attention_explainer import (
    extract_phobert_attention, aggregate_attention,
    cls_token_importance, merge_subword_attention,
    plot_cls_importance_bar, plot_attention_heatmap,
    extract_cross_attention, plot_cross_attention_heatmap,
    plot_patch_importance,
)
from xai.shap_explainer import (
    FusionHeadWrapper, extract_fused_embeddings,
    select_background, compute_shap_values, modality_contribution,
)
from xai.lime_explainer import (
    run_lime_image, run_lime_text,
    save_lime_image_explanation,
)
from xai.case_study import check_sample_artifacts

try:
    from agent import ExplanationAgent
    from agent.config import AgentConfig
    AGENT_AVAILABLE = True
except Exception as _e:
    AGENT_AVAILABLE = False
    print(f'[WARN] AI Agent import failed: {_e}')

set_seed(DEFAULT_SEED)
print('✅ Imports hoàn thành.')

---
## 0.6 · Cấu hình đường dẫn & thiết bị

In [ ]:
device = get_device()
print(f'Device: {device}')

DRIVE_ROOT = '/content/drive/MyDrive'
DEMO_ROOT  = f'{DRIVE_ROOT}/demo_e2e'
XAI_DIR    = f'{DEMO_ROOT}/xai_artifacts'
AGENT_DIR  = f'{DEMO_ROOT}/agent_reports'
for _d in [DEMO_ROOT, XAI_DIR, AGENT_DIR]:
    os.makedirs(_d, exist_ok=True)
print(f'Demo output dir: {DEMO_ROOT}')

# Điều chỉnh nếu Drive path khác
DATA_ROOT = f'{DRIVE_ROOT}/data'
CSV_TEST  = f'{DATA_ROOT}/test.csv'
IMAGE_DIR = f'{DATA_ROOT}/images'
EXP_ID    = BEST_EXP_ID
EXP_DIR   = f'{DRIVE_ROOT}/experiments/{EXP_ID}'

print(f'EXP_DIR : {EXP_DIR}')
print(f'CSV_TEST: {CSV_TEST}')
print(f'IMAGE_DIR: {IMAGE_DIR}')

---
## 0.7 · Tải mô hình, tokenizer, image processor

In [ ]:
print('Đang tải mô hình CrossAttentionFusion...')
model, model_config = load_model(EXP_DIR, device)
model.eval()
print(f'✅ Mô hình: {type(model).__name__}')

try:
    tokenizer = get_tokenizer(model_config.get('text_model_name', BEST_TEXT_MODEL))
except Exception:
    tokenizer = get_tokenizer()
print(f'✅ Tokenizer: {type(tokenizer).__name__}')

try:
    image_processor = get_image_processor(model_config.get('image_model_name', BEST_IMAGE_MODEL))
except Exception:
    image_processor = get_image_processor()
print(f'✅ Image processor: {type(image_processor).__name__}')

target_layer = find_target_layer(model)
print(f'✅ Grad-CAM target layer: {type(target_layer).__name__}')

enable_eager_attention(model)
print('✅ Eager attention enabled cho PhoBERT.')

---
## 0.8 · Hàm tiện ích demo

In [ ]:
# ── run_safe: bao bọc mỗi bước XAI ─────────────────────────────────────────

def run_safe(fn, step_name='XAI step', fallback=None, **kwargs):
    try:
        return fn(**kwargs)
    except Exception as e:
        print(f'  [SKIP] {step_name}: {type(e).__name__}: {e}')
        if os.environ.get('XAI_TRACEBACK'):
            traceback.print_exc()
        return fallback

# ── Bảng dự đoán vs thực tế ─────────────────────────────────────────────────

def display_prediction_table(pred_result, sample_id=''):
    preds = pred_result['predictions']
    gt    = pred_result['ground_truth']
    errs  = pred_result['absolute_errors']
    rows  = []
    for name, disp in zip(TARGET_NAMES, DISPLAY_NAMES):
        rows.append({'Chỉ tiêu': disp,
                     'Thực tế' : f'{gt[name]:.1f}',
                     'Dự đoán' : f'{preds[name]:.1f}',
                     'AE'      : f'{errs[name]:.2f}'})
    df = pd.DataFrame(rows)
    mae = pred_result['mean_mae']
    print(f'\n{"="*50}')
    print(f'  {sample_id}  —  MAE = {mae:.3f}')
    print('='*50)
    print(df.to_string(index=False))
    print('='*50)
    return df

# ── Biểu đồ dự đoán ─────────────────────────────────────────────────────────

def plot_prediction_bars(pred_result, sample_id='', save_path=None):
    preds  = pred_result['predictions']
    gt     = pred_result['ground_truth']
    x      = range(len(TARGET_NAMES))
    p_vals = [preds[n] for n in TARGET_NAMES]
    g_vals = [gt[n]    for n in TARGET_NAMES]
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 0.35
    b_gt   = ax.bar([i - w/2 for i in x], g_vals, w,
                    label='Thực tế', color=COLOR_SCHEMES['bar_gt'], alpha=0.85)
    b_pred = ax.bar([i + w/2 for i in x], p_vals, w,
                    label='Dự đoán', color=COLOR_SCHEMES['bar_pred'], alpha=0.85)
    for b in list(b_gt) + list(b_pred):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(DISPLAY_NAMES, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Điểm (1–10)', fontsize=10)
    ax.set_ylim(0, 12)
    ax.set_title(f'{sample_id} — Dự đoán vs Thực tế', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        print(f'[XAI] Đã lưu: {save_path}')
    plt.show()
    return fig

# ── Hiển thị ảnh review ──────────────────────────────────────────────────────

def show_review_images(sample, sample_id='', max_show=4):
    n = min(sample['num_real_images'], max_show)
    if n == 0:
        print('  (Không có ảnh thực)')
        return
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = sample['loaded_images'][i].convert('RGB').resize((224, 224))
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'Ảnh {i+1}', fontsize=9)
        axes[i].axis('off')
    fig.suptitle(f'{sample_id} — {n} ảnh thực', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ── Dashboard tóm tắt ────────────────────────────────────────────────────────

def print_sample_summary(sample_id, pred_result, artifact_check):
    print(f'\n{chr(9473)*55}')
    print(f'  TÓM TẮT — {sample_id}')
    print(f'{chr(9473)*55}')
    for key, label in [('gradcam','Grad-CAM'),('attention','Attention'),
                        ('cross_attention','Cross-Attention'),('shap','SHAP'),('lime','LIME')]:
        status = '✅' if artifact_check.get(key) else '⚠️ '
        print(f'  {status} {label}')
    print(f"  Completeness: {artifact_check.get('completeness',0)*100:.0f}%")
    print(f"  MAE: {pred_result['mean_mae']:.3f}")
    for n, d in zip(TARGET_NAMES, DISPLAY_NAMES):
        e = pred_result['absolute_errors'][n]
        mark = '✅' if e <= 0.5 else ('⚠️ ' if e <= 1.5 else '❌')
        print(f'    {mark} {d}: AE={e:.2f}')
    print(f'{chr(9473)*55}\n')

# Bảng cross-method (điền dần)
CROSS_METHOD_ROWS = []

def add_cross_method_row(sample_id, pred_result, shap_contrib, lime_text_weights, case_type=''):
    top_words = ', '.join([w for w, _ in lime_text_weights[:3]]) if lime_text_weights else 'N/A'
    CROSS_METHOD_ROWS.append({
        'Mẫu'         : sample_id,
        'Loại'        : case_type,
        'MAE'         : f"{pred_result['mean_mae']:.3f}",
        'SHAP text'   : f"{shap_contrib.get('text_pct',0):.0f}%" if shap_contrib else 'N/A',
        'SHAP image'  : f"{shap_contrib.get('image_pct',0):.0f}%" if shap_contrib else 'N/A',
        'Top LIME'    : top_words,
    })

print('✅ Hàm tiện ích đã định nghĩa.')

---
# 📋 PHẦN 1 — CHỌN MẪU DEMO

| Ký hiệu | Chiến lược | Mô tả |
|---------|-----------|-------|
| **Mẫu A** | Chính xác | MAE ≤ 0.5, tất cả điểm dự đoán gần thực tế |
| **Mẫu B** | Lỗi/xung đột | MAE > 1.5 hoặc có target lệch > 2 điểm |
| **Mẫu C** | Đa ảnh | Có ≥ 3 ảnh thực tế, nội dung review phong phú |

In [ ]:
import ast as _ast

df_test = pd.read_csv(CSV_TEST)
print(f'Tổng mẫu test: {len(df_test)}')

def _count_images(row):
    for col in ['image_paths', 'images', 'image_list']:
        if col in row.index and pd.notna(row[col]):
            try:
                paths = _ast.literal_eval(str(row[col]))
                return len([p for p in paths if p and str(p).strip()])
            except Exception:
                pass
    return 0

df_test['_n_images'] = df_test.apply(_count_images, axis=1)

print('Đang scan MAE (tối đa 200 mẫu) ...')
SCAN_MAX = min(200, len(df_test))
scan_results = []
for idx in range(SCAN_MAX):
    try:
        samp = load_single_sample(CSV_TEST, idx, tokenizer, image_processor, IMAGE_DIR, device)
        pred = get_prediction(model, samp)
        scan_results.append({'idx': idx, 'mae': pred['mean_mae'],
                              'n_images': samp['num_real_images'],
                              'text_len': len(samp.get('text', ''))})
    except Exception:
        scan_results.append({'idx': idx, 'mae': None, 'n_images': 0, 'text_len': 0})

df_scan = pd.DataFrame(scan_results)
df_scan = df_scan[df_scan['mae'].notna()]
print(f'Scan xong {len(df_scan)} mẫu hợp lệ.')
print(f'MAE range: {df_scan["mae"].min():.3f} — {df_scan["mae"].max():.3f}')

In [ ]:
# Mẫu A: MAE thấp (chính xác)
df_accurate = df_scan[df_scan['mae'] <= df_scan['mae'].quantile(0.10)]
IDX_A = int(df_accurate.sort_values('mae').iloc[0]['idx'])

# Mẫu B: MAE cao (lỗi)
df_error = df_scan[df_scan['mae'] >= df_scan['mae'].quantile(0.90)]
IDX_B = int(df_error.sort_values('mae', ascending=False).iloc[0]['idx'])

# Mẫu C: Nhiều ảnh nhất + MAE trung bình
df_multi = df_scan[df_scan['n_images'] >= 3].sort_values('n_images', ascending=False)
if len(df_multi) > 0:
    median_mae = df_multi['mae'].median()
    df_multi = df_multi.copy()
    df_multi['_dist'] = (df_multi['mae'] - median_mae).abs()
    IDX_C = int(df_multi.sort_values('_dist').iloc[0]['idx'])
else:
    IDX_C = int(df_scan.sort_values('text_len', ascending=False).iloc[0]['idx'])

# Đảm bảo A, B, C khác nhau
if IDX_B == IDX_A:
    IDX_B = int(df_error.sort_values('mae', ascending=False).iloc[1]['idx'])
if IDX_C in (IDX_A, IDX_B):
    candidates = df_scan[~df_scan['idx'].isin([IDX_A, IDX_B])]
    IDX_C = int(candidates.sort_values('n_images', ascending=False).iloc[0]['idx'])

SAMPLE_IDS     = {'A': f'sample_{IDX_A:04d}', 'B': f'sample_{IDX_B:04d}', 'C': f'sample_{IDX_C:04d}'}
SAMPLE_INDICES = {'A': IDX_A, 'B': IDX_B, 'C': IDX_C}
CASE_TYPES     = {'A': 'accurate', 'B': 'error', 'C': 'multimodal'}

print('╔══════════════════════════════════════╗')
print('║       DANH SÁCH MẪU DEMO            ║')
print('╠══════════════════════════════════════╣')
for k in ['A', 'B', 'C']:
    row = df_scan[df_scan['idx'] == SAMPLE_INDICES[k]]
    mae_s = f"{row['mae'].values[0]:.3f}" if len(row) else '?'
    n_s   = str(int(row['n_images'].values[0])) if len(row) else '?'
    print(f'║  Mẫu {k}: idx={SAMPLE_INDICES[k]:4d} | MAE={mae_s} | ảnh={n_s}    ║')
print('╚══════════════════════════════════════╝')

---
# 🧪 PHẦN 2 — MẪU A: DỰ ĐOÁN CHÍNH XÁC

## 2.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu A và dự đoán ────────────────────────────────────────────────
SID_A  = SAMPLE_IDS['A']
IDX_A  = SAMPLE_INDICES['A']
CASE_A = CASE_TYPES['A']

xai_dir_A   = f'{XAI_DIR}/{SID_A}'
agent_dir_A = f'{AGENT_DIR}/{SID_A}'
for _d in [xai_dir_A, agent_dir_A]:
    os.makedirs(_d, exist_ok=True)

print(f'Đang tải {SID_A} (idx={IDX_A}) ...')
sample_A = load_single_sample(
    CSV_TEST, IDX_A, tokenizer, image_processor, IMAGE_DIR, device
)
print(f'  Text ({len(sample_A["text"])} ký tự): {sample_A["text"][:150]} ...')
print(f'  Số ảnh thực: {sample_A["num_real_images"]}')
show_review_images(sample_A, SID_A)

pred_result_A = get_prediction(model, sample_A)
display_prediction_table(pred_result_A, SID_A)
plot_prediction_bars(
    pred_result_A, SID_A,
    save_path=f'{xai_dir_A}/{SID_A}_prediction.png'
)

## 2.2 · Grad-CAM — Vùng ảnh quan trọng

> Shared encoder → cosine sim >0.95 across 5 targets → chỉ hiển thị Overall Satisfaction.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
TARGET_IDX_GRADCAM = 4
gradcam_results_A = {}
for img_idx in range(min(sample_A['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_A, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_A[img_idx] = cam

n_show = min(sample_A['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_A['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_A.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{xai_dir_A}/gradcam/{SID_A}_gradcam_img{img_idx}_overall.png'
                os.makedirs(os.path.dirname(cam_save), exist_ok=True)
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_A} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{xai_dir_A}/{SID_A}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

## 2.3 · PhoBERT Attention — Từ quan trọng trong review

In [ ]:
# ── PhoBERT Attention: CLS → top words ───────────────────────────────────────
attn_result_A = run_safe(
    extract_phobert_attention, step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_A['input_ids'],
    attention_mask=sample_A['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_A = []

if attn_result_A is not None:
    attentions_A = attn_result_A['attentions']
    tokens_A     = attn_result_A['tokens']
    seq_len_A    = attn_result_A['seq_len']
    print(f'  tokens={seq_len_A}, attn shape={attentions_A.shape}')

    agg_matrix_A      = aggregate_attention(attentions_A, strategy='last_layer_mean')
    cls_result_A      = cls_token_importance(agg_matrix_A, tokens_A)
    word_importances_A = merge_subword_attention(
        cls_result_A['importances'], tokens_A, strategy='mean')

    print(f'Top 10 từ ({SID_A}):')
    for word, score in word_importances_A[:10]:
        print(f'  {word:<20s} {score:.4f}')

    w_tok = [w for w, _ in word_importances_A]
    w_val = [v for _, v in word_importances_A]
    bar_path = f'{xai_dir_A}/attention/{SID_A}_cls_word_importance.png'
    os.makedirs(os.path.dirname(bar_path), exist_ok=True)
    plot_cls_importance_bar(tokens=w_tok, importances=w_val,
                            title=f'PhoBERT CLS Attention — {SID_A}',
                            save_path=bar_path, top_k=15, dpi=DEFAULT_DPI)
    plt.show()

    if seq_len_A <= 60:
        hm_path = f'{xai_dir_A}/attention/{SID_A}_attention_heatmap.png'
        plot_attention_heatmap(
            attention_matrix=agg_matrix_A, tokens=tokens_A,
            title=f'Attention Heatmap — {SID_A}',
            save_path=hm_path, dpi=DEFAULT_DPI)
        plt.show()

    imp_json = f'{xai_dir_A}/attention/{SID_A}_word_importance.json'
    with open(imp_json, 'w', encoding='utf-8') as f:
        json.dump(word_importances_A, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {imp_json}')
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 2.4 · Cross-Attention — Tương tác Text ↔ Image

> **T2I (Text→Image):** token nào attend nhiều patch nào
> **I2T (Image→Text):** patch nào attend nhiều token nào

In [ ]:
# ── Bidirectional Cross-Attention T2I + I2T ───────────────────────────────────
cross_result_A = run_safe(
    extract_cross_attention, step_name='extract_cross_attention',
    fallback=None,
    model=model, sample=sample_A, tokenizer=tokenizer,
)

t2i_A = None
i2t_A = None

if cross_result_A is not None:
    t2i_A      = cross_result_A['t2i_attn']
    i2t_A      = cross_result_A['i2t_attn']
    ca_tokens_A = cross_result_A['tokens']
    T, P = t2i_A.shape
    H = W = int(P ** 0.5)
    print(f'  T2I={t2i_A.shape}, I2T={i2t_A.shape}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # T2I heatmap
    patch_labels = [f'{r},{c}' for r in range(H) for c in range(W)][:P]
    _tkns = ca_tokens_A[:min(T, 40)]
    _t2i  = t2i_A[:min(T, 40), :]
    try:
        import seaborn as sns
        sns.heatmap(_t2i, xticklabels=patch_labels, yticklabels=_tkns,
                    cmap='viridis', ax=axes[0], cbar_kws={'shrink': 0.6})
    except ImportError:
        axes[0].imshow(_t2i, aspect='auto', cmap='viridis')
    axes[0].set_title(f'T2I: Text→Image ({SID_A})', fontsize=10, fontweight='bold')
    axes[0].set_xlabel(f'Image Patches ({H}×{W})', fontsize=9)
    axes[0].set_ylabel('Text Tokens', fontsize=9)
    axes[0].tick_params(axis='both', labelsize=5)

    # I2T patch importance overlay
    if sample_A['num_real_images'] > 0:
        import io as _io
        from PIL import Image as _PILImg
        fig_i2t = run_safe(plot_patch_importance, step_name='patch_importance',
                           fallback=None, i2t_attn=i2t_A,
                           original_image=sample_A['loaded_images'][0],
                           title=f'I2T Patch Importance — {SID_A}')
        if fig_i2t is not None:
            buf = _io.BytesIO()
            fig_i2t.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            axes[1].imshow(np.array(_PILImg.open(buf)))
            axes[1].axis('off')
            axes[1].set_title(f'I2T: Patch Importance — {SID_A}', fontsize=10)
            plt.close(fig_i2t)
        else:
            axes[1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                         transform=axes[1].transAxes); axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'Không có ảnh', ha='center', va='center',
                     transform=axes[1].transAxes); axes[1].axis('off')

    fig.suptitle(f'Bidirectional Cross-Attention — {SID_A}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    ca_path = f'{xai_dir_A}/cross_attention/{SID_A}_cross_attention.png'
    os.makedirs(os.path.dirname(ca_path), exist_ok=True)
    fig.savefig(ca_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {ca_path}')
    np.savez(f'{xai_dir_A}/cross_attention/{SID_A}_cross_attn.npz',
             t2i=t2i_A, i2t=i2t_A)
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 2.5 · SHAP — Đóng góp fused embedding [1024]

> dims 0:512 = text-origin, dims 512:1024 = image-origin (cả hai đã qua cross-attention).

In [ ]:
# ── SHAP DeepExplainer on fused embedding [1024] ─────────────────────────────
# FUSED_DIM=1024: dims 0:512=text-origin, 512:1024=image-origin (cross-attended)

class _SingleSampleDL_A:
    def __init__(self, s):
        self.s = s
    def __iter__(self):
        s = self.s
        yield {
            'input_ids'     : s['input_ids'],
            'attention_mask': s['attention_mask'],
            'pixel_values'  : s['pixel_values'],
            'num_images'    : s['num_images'].unsqueeze(0) if s['num_images'].dim()==0 else s['num_images'],
            'labels'        : s['factor_scores'].unsqueeze(0),
        }

fused_A, _, _ = run_safe(
    extract_fused_embeddings, step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model, dataloader=_SingleSampleDL_A(sample_A), device=device, max_samples=1,
)

shap_contrib_A = None
shap_vals_A    = None

if fused_A is not None:
    wrapper_A = FusionHeadWrapper(model.head, score_index=4)
    print('[SHAP] Đang tính SHAP values ...')
    shap_res_A = run_safe(
        compute_shap_values, step_name='compute_shap_values',
        fallback=(None, None),
        wrapper=wrapper_A, background=fused_A, samples=fused_A,
    )
    if shap_res_A is not None and shap_res_A[0] is not None:
        shap_vals_A, base_val_A = shap_res_A
        shap_contrib_A = modality_contribution(shap_vals_A[0])
        print(f'  Text-origin : {shap_contrib_A["text_pct"]:.1f}%')
        print(f'  Image-origin: {shap_contrib_A["image_pct"]:.1f}%')

        sv_flat = shap_vals_A[0]
        top_idx = np.argsort(np.abs(sv_flat))[-20:][::-1]
        top_sv  = sv_flat[top_idx]
        dim_labels = [f'T{i}' if i < 512 else f'I{i-512}' for i in top_idx]
        shap_colors = [COLOR_SCHEMES['shap_positive'] if v >= 0
                       else COLOR_SCHEMES['shap_negative'] for v in top_sv]

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        pie_vals = [shap_contrib_A['text_abs'], shap_contrib_A['image_abs']]
        pie_labs = [f"Text-origin\n{shap_contrib_A['text_pct']:.0f}%",
                    f"Image-origin\n{shap_contrib_A['image_pct']:.0f}%"]
        pie_cols = [COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image']]
        axes[0].pie(pie_vals, labels=pie_labs, colors=pie_cols,
                    autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
        axes[0].set_title(f'SHAP Modality Contribution\n{SID_A}',
                           fontsize=11, fontweight='bold')

        axes[1].barh(range(20), top_sv[::-1], color=shap_colors[::-1])
        axes[1].set_yticks(range(20))
        axes[1].set_yticklabels(dim_labels[::-1], fontsize=7)
        axes[1].axvline(0, color='black', lw=0.8)
        axes[1].set_xlabel('SHAP value', fontsize=9)
        axes[1].set_title(f'Top-20 SHAP dims\n(T=text, I=image)', fontsize=11, fontweight='bold')

        fig.suptitle(f'SHAP Analysis — {SID_A} (Overall Satisfaction)',
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        shap_path = f'{xai_dir_A}/shap/{SID_A}_shap_analysis.png'
        os.makedirs(os.path.dirname(shap_path), exist_ok=True)
        fig.savefig(shap_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'[XAI] Đã lưu: {shap_path}')

        contrib_path = f'{xai_dir_A}/shap/{SID_A}_shap_contribution.json'
        with open(contrib_path, 'w') as f:
            json.dump(shap_contrib_A, f, indent=2)
    else:
        print('[SKIP] compute_shap_values thất bại.')
else:
    print('[SKIP] Không extract được fused embeddings.')

## 2.6 · LIME — Giải thích cục bộ (Text + Image)

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_A = {}
lime_text_exp_A     = None
lime_image_exp_A    = None
lime_img_paths_A    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_A = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_A, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_A = []
if lime_text_exp_A is not None:
    raw_weights_A = lime_text_exp_A.as_list(label=1)
    lime_text_weights_A = dict(raw_weights_A)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_A, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lt_path = f'{xai_dir_A}/lime/{SID_A}_lime_text_weights.json'
    os.makedirs(os.path.dirname(lt_path), exist_ok=True)
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_A, f, ensure_ascii=False, indent=2)
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_A['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_A = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_A, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_A is not None:
        lime_img_paths_A = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_A,
            original_image=sample_A['loaded_images'][0],
            save_dir=f'{xai_dir_A}/lime',
            sample_id=SID_A,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_A:
    w_sorted = sorted(lime_text_weights_A.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_A.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_A} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{xai_dir_A}/{SID_A}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')

## 2.7 · AI Agent — Báo cáo tổng hợp

> AI Agent (GPT-4o) phân tích evidence XAI và viết báo cáo tiếng Việt.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_A = check_sample_artifacts(SID_A, XAI_DIR)

add_cross_method_row(
    SID_A, pred_result_A,
    shap_contrib_A,
    raw_weights_A,
    case_type=CASE_A,
)

agent_output_A = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_A} ...')
        agent_A = ExplanationAgent(agent_config)
        agent_output_A = run_safe(
            agent_A.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_A,
            review_text=sample_A['text'],
            predictions=pred_result_A['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_A['ground_truth'],
            case_type=CASE_A,
            language='vi',
            mode='full',
            num_images=sample_A['num_real_images'],
            output_dir=agent_dir_A,
        )
        if agent_output_A:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_A.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print("  → os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_A, pred_result_A, artifact_check_A)
print(f'✅ HOÀN THÀNH MẪU A: {SID_A}')

---
# 🧪 PHẦN 3 — MẪU B: LỖI / XUNG ĐỘT

## 3.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu B và dự đoán ────────────────────────────────────────────────
SID_B  = SAMPLE_IDS['B']
IDX_B  = SAMPLE_INDICES['B']
CASE_B = CASE_TYPES['B']

xai_dir_B   = f'{XAI_DIR}/{SID_B}'
agent_dir_B = f'{AGENT_DIR}/{SID_B}'
for _d in [xai_dir_B, agent_dir_B]:
    os.makedirs(_d, exist_ok=True)

print(f'Đang tải {SID_B} (idx={IDX_B}) ...')
sample_B = load_single_sample(
    CSV_TEST, IDX_B, tokenizer, image_processor, IMAGE_DIR, device
)
print(f'  Text ({len(sample_B["text"])} ký tự): {sample_B["text"][:150]} ...')
print(f'  Số ảnh thực: {sample_B["num_real_images"]}')
show_review_images(sample_B, SID_B)

pred_result_B = get_prediction(model, sample_B)
display_prediction_table(pred_result_B, SID_B)
plot_prediction_bars(
    pred_result_B, SID_B,
    save_path=f'{xai_dir_B}/{SID_B}_prediction.png'
)

## 3.2 · Grad-CAM — Vùng ảnh quan trọng

> Shared encoder → cosine sim >0.95 across 5 targets → chỉ hiển thị Overall Satisfaction.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
TARGET_IDX_GRADCAM = 4
gradcam_results_B = {}
for img_idx in range(min(sample_B['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_B, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_B[img_idx] = cam

n_show = min(sample_B['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_B['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_B.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{xai_dir_B}/gradcam/{SID_B}_gradcam_img{img_idx}_overall.png'
                os.makedirs(os.path.dirname(cam_save), exist_ok=True)
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_B} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{xai_dir_B}/{SID_B}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

## 3.3 · PhoBERT Attention — Từ quan trọng trong review

In [ ]:
# ── PhoBERT Attention: CLS → top words ───────────────────────────────────────
attn_result_B = run_safe(
    extract_phobert_attention, step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_B['input_ids'],
    attention_mask=sample_B['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_B = []

if attn_result_B is not None:
    attentions_B = attn_result_B['attentions']
    tokens_B     = attn_result_B['tokens']
    seq_len_B    = attn_result_B['seq_len']
    print(f'  tokens={seq_len_B}, attn shape={attentions_B.shape}')

    agg_matrix_B      = aggregate_attention(attentions_B, strategy='last_layer_mean')
    cls_result_B      = cls_token_importance(agg_matrix_B, tokens_B)
    word_importances_B = merge_subword_attention(
        cls_result_B['importances'], tokens_B, strategy='mean')

    print(f'Top 10 từ ({SID_B}):')
    for word, score in word_importances_B[:10]:
        print(f'  {word:<20s} {score:.4f}')

    w_tok = [w for w, _ in word_importances_B]
    w_val = [v for _, v in word_importances_B]
    bar_path = f'{xai_dir_B}/attention/{SID_B}_cls_word_importance.png'
    os.makedirs(os.path.dirname(bar_path), exist_ok=True)
    plot_cls_importance_bar(tokens=w_tok, importances=w_val,
                            title=f'PhoBERT CLS Attention — {SID_B}',
                            save_path=bar_path, top_k=15, dpi=DEFAULT_DPI)
    plt.show()

    if seq_len_B <= 60:
        hm_path = f'{xai_dir_B}/attention/{SID_B}_attention_heatmap.png'
        plot_attention_heatmap(
            attention_matrix=agg_matrix_B, tokens=tokens_B,
            title=f'Attention Heatmap — {SID_B}',
            save_path=hm_path, dpi=DEFAULT_DPI)
        plt.show()

    imp_json = f'{xai_dir_B}/attention/{SID_B}_word_importance.json'
    with open(imp_json, 'w', encoding='utf-8') as f:
        json.dump(word_importances_B, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {imp_json}')
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 3.4 · Cross-Attention — Tương tác Text ↔ Image

> **T2I (Text→Image):** token nào attend nhiều patch nào
> **I2T (Image→Text):** patch nào attend nhiều token nào

In [ ]:
# ── Bidirectional Cross-Attention T2I + I2T ───────────────────────────────────
cross_result_B = run_safe(
    extract_cross_attention, step_name='extract_cross_attention',
    fallback=None,
    model=model, sample=sample_B, tokenizer=tokenizer,
)

t2i_B = None
i2t_B = None

if cross_result_B is not None:
    t2i_B      = cross_result_B['t2i_attn']
    i2t_B      = cross_result_B['i2t_attn']
    ca_tokens_B = cross_result_B['tokens']
    T, P = t2i_B.shape
    H = W = int(P ** 0.5)
    print(f'  T2I={t2i_B.shape}, I2T={i2t_B.shape}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # T2I heatmap
    patch_labels = [f'{r},{c}' for r in range(H) for c in range(W)][:P]
    _tkns = ca_tokens_B[:min(T, 40)]
    _t2i  = t2i_B[:min(T, 40), :]
    try:
        import seaborn as sns
        sns.heatmap(_t2i, xticklabels=patch_labels, yticklabels=_tkns,
                    cmap='viridis', ax=axes[0], cbar_kws={'shrink': 0.6})
    except ImportError:
        axes[0].imshow(_t2i, aspect='auto', cmap='viridis')
    axes[0].set_title(f'T2I: Text→Image ({SID_B})', fontsize=10, fontweight='bold')
    axes[0].set_xlabel(f'Image Patches ({H}×{W})', fontsize=9)
    axes[0].set_ylabel('Text Tokens', fontsize=9)
    axes[0].tick_params(axis='both', labelsize=5)

    # I2T patch importance overlay
    if sample_B['num_real_images'] > 0:
        import io as _io
        from PIL import Image as _PILImg
        fig_i2t = run_safe(plot_patch_importance, step_name='patch_importance',
                           fallback=None, i2t_attn=i2t_B,
                           original_image=sample_B['loaded_images'][0],
                           title=f'I2T Patch Importance — {SID_B}')
        if fig_i2t is not None:
            buf = _io.BytesIO()
            fig_i2t.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            axes[1].imshow(np.array(_PILImg.open(buf)))
            axes[1].axis('off')
            axes[1].set_title(f'I2T: Patch Importance — {SID_B}', fontsize=10)
            plt.close(fig_i2t)
        else:
            axes[1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                         transform=axes[1].transAxes); axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'Không có ảnh', ha='center', va='center',
                     transform=axes[1].transAxes); axes[1].axis('off')

    fig.suptitle(f'Bidirectional Cross-Attention — {SID_B}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    ca_path = f'{xai_dir_B}/cross_attention/{SID_B}_cross_attention.png'
    os.makedirs(os.path.dirname(ca_path), exist_ok=True)
    fig.savefig(ca_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {ca_path}')
    np.savez(f'{xai_dir_B}/cross_attention/{SID_B}_cross_attn.npz',
             t2i=t2i_B, i2t=i2t_B)
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 3.5 · SHAP — Đóng góp fused embedding [1024]

> dims 0:512 = text-origin, dims 512:1024 = image-origin (cả hai đã qua cross-attention).

In [ ]:
# ── SHAP DeepExplainer on fused embedding [1024] ─────────────────────────────
# FUSED_DIM=1024: dims 0:512=text-origin, 512:1024=image-origin (cross-attended)

class _SingleSampleDL_B:
    def __init__(self, s):
        self.s = s
    def __iter__(self):
        s = self.s
        yield {
            'input_ids'     : s['input_ids'],
            'attention_mask': s['attention_mask'],
            'pixel_values'  : s['pixel_values'],
            'num_images'    : s['num_images'].unsqueeze(0) if s['num_images'].dim()==0 else s['num_images'],
            'labels'        : s['factor_scores'].unsqueeze(0),
        }

fused_B, _, _ = run_safe(
    extract_fused_embeddings, step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model, dataloader=_SingleSampleDL_B(sample_B), device=device, max_samples=1,
)

shap_contrib_B = None
shap_vals_B    = None

if fused_B is not None:
    wrapper_B = FusionHeadWrapper(model.head, score_index=4)
    print('[SHAP] Đang tính SHAP values ...')
    shap_res_B = run_safe(
        compute_shap_values, step_name='compute_shap_values',
        fallback=(None, None),
        wrapper=wrapper_B, background=fused_B, samples=fused_B,
    )
    if shap_res_B is not None and shap_res_B[0] is not None:
        shap_vals_B, base_val_B = shap_res_B
        shap_contrib_B = modality_contribution(shap_vals_B[0])
        print(f'  Text-origin : {shap_contrib_B["text_pct"]:.1f}%')
        print(f'  Image-origin: {shap_contrib_B["image_pct"]:.1f}%')

        sv_flat = shap_vals_B[0]
        top_idx = np.argsort(np.abs(sv_flat))[-20:][::-1]
        top_sv  = sv_flat[top_idx]
        dim_labels = [f'T{i}' if i < 512 else f'I{i-512}' for i in top_idx]
        shap_colors = [COLOR_SCHEMES['shap_positive'] if v >= 0
                       else COLOR_SCHEMES['shap_negative'] for v in top_sv]

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        pie_vals = [shap_contrib_B['text_abs'], shap_contrib_B['image_abs']]
        pie_labs = [f"Text-origin\n{shap_contrib_B['text_pct']:.0f}%",
                    f"Image-origin\n{shap_contrib_B['image_pct']:.0f}%"]
        pie_cols = [COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image']]
        axes[0].pie(pie_vals, labels=pie_labs, colors=pie_cols,
                    autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
        axes[0].set_title(f'SHAP Modality Contribution\n{SID_B}',
                           fontsize=11, fontweight='bold')

        axes[1].barh(range(20), top_sv[::-1], color=shap_colors[::-1])
        axes[1].set_yticks(range(20))
        axes[1].set_yticklabels(dim_labels[::-1], fontsize=7)
        axes[1].axvline(0, color='black', lw=0.8)
        axes[1].set_xlabel('SHAP value', fontsize=9)
        axes[1].set_title(f'Top-20 SHAP dims\n(T=text, I=image)', fontsize=11, fontweight='bold')

        fig.suptitle(f'SHAP Analysis — {SID_B} (Overall Satisfaction)',
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        shap_path = f'{xai_dir_B}/shap/{SID_B}_shap_analysis.png'
        os.makedirs(os.path.dirname(shap_path), exist_ok=True)
        fig.savefig(shap_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'[XAI] Đã lưu: {shap_path}')

        contrib_path = f'{xai_dir_B}/shap/{SID_B}_shap_contribution.json'
        with open(contrib_path, 'w') as f:
            json.dump(shap_contrib_B, f, indent=2)
    else:
        print('[SKIP] compute_shap_values thất bại.')
else:
    print('[SKIP] Không extract được fused embeddings.')

## 3.6 · LIME — Giải thích cục bộ (Text + Image)

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_B = {}
lime_text_exp_B     = None
lime_image_exp_B    = None
lime_img_paths_B    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_B = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_B, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_B = []
if lime_text_exp_B is not None:
    raw_weights_B = lime_text_exp_B.as_list(label=1)
    lime_text_weights_B = dict(raw_weights_B)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_B, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lt_path = f'{xai_dir_B}/lime/{SID_B}_lime_text_weights.json'
    os.makedirs(os.path.dirname(lt_path), exist_ok=True)
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_B, f, ensure_ascii=False, indent=2)
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_B['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_B = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_B, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_B is not None:
        lime_img_paths_B = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_B,
            original_image=sample_B['loaded_images'][0],
            save_dir=f'{xai_dir_B}/lime',
            sample_id=SID_B,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_B:
    w_sorted = sorted(lime_text_weights_B.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_B.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_B} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{xai_dir_B}/{SID_B}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')

## 3.7 · AI Agent — Báo cáo tổng hợp

> AI Agent (GPT-4o) phân tích evidence XAI và viết báo cáo tiếng Việt.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_B = check_sample_artifacts(SID_B, XAI_DIR)

add_cross_method_row(
    SID_B, pred_result_B,
    shap_contrib_B,
    raw_weights_B,
    case_type=CASE_B,
)

agent_output_B = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_B} ...')
        agent_B = ExplanationAgent(agent_config)
        agent_output_B = run_safe(
            agent_B.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_B,
            review_text=sample_B['text'],
            predictions=pred_result_B['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_B['ground_truth'],
            case_type=CASE_B,
            language='vi',
            mode='full',
            num_images=sample_B['num_real_images'],
            output_dir=agent_dir_B,
        )
        if agent_output_B:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_B.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print("  → os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_B, pred_result_B, artifact_check_B)
print(f'✅ HOÀN THÀNH MẪU B: {SID_B}')

---
# 🧪 PHẦN 4 — MẪU C: ĐA ẢNH PHONG PHÚ

## 4.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu C và dự đoán ────────────────────────────────────────────────
SID_C  = SAMPLE_IDS['C']
IDX_C  = SAMPLE_INDICES['C']
CASE_C = CASE_TYPES['C']

xai_dir_C   = f'{XAI_DIR}/{SID_C}'
agent_dir_C = f'{AGENT_DIR}/{SID_C}'
for _d in [xai_dir_C, agent_dir_C]:
    os.makedirs(_d, exist_ok=True)

print(f'Đang tải {SID_C} (idx={IDX_C}) ...')
sample_C = load_single_sample(
    CSV_TEST, IDX_C, tokenizer, image_processor, IMAGE_DIR, device
)
print(f'  Text ({len(sample_C["text"])} ký tự): {sample_C["text"][:150]} ...')
print(f'  Số ảnh thực: {sample_C["num_real_images"]}')
show_review_images(sample_C, SID_C)

pred_result_C = get_prediction(model, sample_C)
display_prediction_table(pred_result_C, SID_C)
plot_prediction_bars(
    pred_result_C, SID_C,
    save_path=f'{xai_dir_C}/{SID_C}_prediction.png'
)

## 4.2 · Grad-CAM — Vùng ảnh quan trọng

> Shared encoder → cosine sim >0.95 across 5 targets → chỉ hiển thị Overall Satisfaction.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
TARGET_IDX_GRADCAM = 4
gradcam_results_C = {}
for img_idx in range(min(sample_C['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_C, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_C[img_idx] = cam

n_show = min(sample_C['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_C['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_C.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{xai_dir_C}/gradcam/{SID_C}_gradcam_img{img_idx}_overall.png'
                os.makedirs(os.path.dirname(cam_save), exist_ok=True)
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_C} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{xai_dir_C}/{SID_C}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

## 4.3 · PhoBERT Attention — Từ quan trọng trong review

In [ ]:
# ── PhoBERT Attention: CLS → top words ───────────────────────────────────────
attn_result_C = run_safe(
    extract_phobert_attention, step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_C['input_ids'],
    attention_mask=sample_C['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_C = []

if attn_result_C is not None:
    attentions_C = attn_result_C['attentions']
    tokens_C     = attn_result_C['tokens']
    seq_len_C    = attn_result_C['seq_len']
    print(f'  tokens={seq_len_C}, attn shape={attentions_C.shape}')

    agg_matrix_C      = aggregate_attention(attentions_C, strategy='last_layer_mean')
    cls_result_C      = cls_token_importance(agg_matrix_C, tokens_C)
    word_importances_C = merge_subword_attention(
        cls_result_C['importances'], tokens_C, strategy='mean')

    print(f'Top 10 từ ({SID_C}):')
    for word, score in word_importances_C[:10]:
        print(f'  {word:<20s} {score:.4f}')

    w_tok = [w for w, _ in word_importances_C]
    w_val = [v for _, v in word_importances_C]
    bar_path = f'{xai_dir_C}/attention/{SID_C}_cls_word_importance.png'
    os.makedirs(os.path.dirname(bar_path), exist_ok=True)
    plot_cls_importance_bar(tokens=w_tok, importances=w_val,
                            title=f'PhoBERT CLS Attention — {SID_C}',
                            save_path=bar_path, top_k=15, dpi=DEFAULT_DPI)
    plt.show()

    if seq_len_C <= 60:
        hm_path = f'{xai_dir_C}/attention/{SID_C}_attention_heatmap.png'
        plot_attention_heatmap(
            attention_matrix=agg_matrix_C, tokens=tokens_C,
            title=f'Attention Heatmap — {SID_C}',
            save_path=hm_path, dpi=DEFAULT_DPI)
        plt.show()

    imp_json = f'{xai_dir_C}/attention/{SID_C}_word_importance.json'
    with open(imp_json, 'w', encoding='utf-8') as f:
        json.dump(word_importances_C, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {imp_json}')
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 4.4 · Cross-Attention — Tương tác Text ↔ Image

> **T2I (Text→Image):** token nào attend nhiều patch nào
> **I2T (Image→Text):** patch nào attend nhiều token nào

In [ ]:
# ── Bidirectional Cross-Attention T2I + I2T ───────────────────────────────────
cross_result_C = run_safe(
    extract_cross_attention, step_name='extract_cross_attention',
    fallback=None,
    model=model, sample=sample_C, tokenizer=tokenizer,
)

t2i_C = None
i2t_C = None

if cross_result_C is not None:
    t2i_C      = cross_result_C['t2i_attn']
    i2t_C      = cross_result_C['i2t_attn']
    ca_tokens_C = cross_result_C['tokens']
    T, P = t2i_C.shape
    H = W = int(P ** 0.5)
    print(f'  T2I={t2i_C.shape}, I2T={i2t_C.shape}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # T2I heatmap
    patch_labels = [f'{r},{c}' for r in range(H) for c in range(W)][:P]
    _tkns = ca_tokens_C[:min(T, 40)]
    _t2i  = t2i_C[:min(T, 40), :]
    try:
        import seaborn as sns
        sns.heatmap(_t2i, xticklabels=patch_labels, yticklabels=_tkns,
                    cmap='viridis', ax=axes[0], cbar_kws={'shrink': 0.6})
    except ImportError:
        axes[0].imshow(_t2i, aspect='auto', cmap='viridis')
    axes[0].set_title(f'T2I: Text→Image ({SID_C})', fontsize=10, fontweight='bold')
    axes[0].set_xlabel(f'Image Patches ({H}×{W})', fontsize=9)
    axes[0].set_ylabel('Text Tokens', fontsize=9)
    axes[0].tick_params(axis='both', labelsize=5)

    # I2T patch importance overlay
    if sample_C['num_real_images'] > 0:
        import io as _io
        from PIL import Image as _PILImg
        fig_i2t = run_safe(plot_patch_importance, step_name='patch_importance',
                           fallback=None, i2t_attn=i2t_C,
                           original_image=sample_C['loaded_images'][0],
                           title=f'I2T Patch Importance — {SID_C}')
        if fig_i2t is not None:
            buf = _io.BytesIO()
            fig_i2t.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            axes[1].imshow(np.array(_PILImg.open(buf)))
            axes[1].axis('off')
            axes[1].set_title(f'I2T: Patch Importance — {SID_C}', fontsize=10)
            plt.close(fig_i2t)
        else:
            axes[1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                         transform=axes[1].transAxes); axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'Không có ảnh', ha='center', va='center',
                     transform=axes[1].transAxes); axes[1].axis('off')

    fig.suptitle(f'Bidirectional Cross-Attention — {SID_C}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    ca_path = f'{xai_dir_C}/cross_attention/{SID_C}_cross_attention.png'
    os.makedirs(os.path.dirname(ca_path), exist_ok=True)
    fig.savefig(ca_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {ca_path}')
    np.savez(f'{xai_dir_C}/cross_attention/{SID_C}_cross_attn.npz',
             t2i=t2i_C, i2t=i2t_C)
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 4.5 · SHAP — Đóng góp fused embedding [1024]

> dims 0:512 = text-origin, dims 512:1024 = image-origin (cả hai đã qua cross-attention).

In [ ]:
# ── SHAP DeepExplainer on fused embedding [1024] ─────────────────────────────
# FUSED_DIM=1024: dims 0:512=text-origin, 512:1024=image-origin (cross-attended)

class _SingleSampleDL_C:
    def __init__(self, s):
        self.s = s
    def __iter__(self):
        s = self.s
        yield {
            'input_ids'     : s['input_ids'],
            'attention_mask': s['attention_mask'],
            'pixel_values'  : s['pixel_values'],
            'num_images'    : s['num_images'].unsqueeze(0) if s['num_images'].dim()==0 else s['num_images'],
            'labels'        : s['factor_scores'].unsqueeze(0),
        }

fused_C, _, _ = run_safe(
    extract_fused_embeddings, step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model, dataloader=_SingleSampleDL_C(sample_C), device=device, max_samples=1,
)

shap_contrib_C = None
shap_vals_C    = None

if fused_C is not None:
    wrapper_C = FusionHeadWrapper(model.head, score_index=4)
    print('[SHAP] Đang tính SHAP values ...')
    shap_res_C = run_safe(
        compute_shap_values, step_name='compute_shap_values',
        fallback=(None, None),
        wrapper=wrapper_C, background=fused_C, samples=fused_C,
    )
    if shap_res_C is not None and shap_res_C[0] is not None:
        shap_vals_C, base_val_C = shap_res_C
        shap_contrib_C = modality_contribution(shap_vals_C[0])
        print(f'  Text-origin : {shap_contrib_C["text_pct"]:.1f}%')
        print(f'  Image-origin: {shap_contrib_C["image_pct"]:.1f}%')

        sv_flat = shap_vals_C[0]
        top_idx = np.argsort(np.abs(sv_flat))[-20:][::-1]
        top_sv  = sv_flat[top_idx]
        dim_labels = [f'T{i}' if i < 512 else f'I{i-512}' for i in top_idx]
        shap_colors = [COLOR_SCHEMES['shap_positive'] if v >= 0
                       else COLOR_SCHEMES['shap_negative'] for v in top_sv]

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        pie_vals = [shap_contrib_C['text_abs'], shap_contrib_C['image_abs']]
        pie_labs = [f"Text-origin\n{shap_contrib_C['text_pct']:.0f}%",
                    f"Image-origin\n{shap_contrib_C['image_pct']:.0f}%"]
        pie_cols = [COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image']]
        axes[0].pie(pie_vals, labels=pie_labs, colors=pie_cols,
                    autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
        axes[0].set_title(f'SHAP Modality Contribution\n{SID_C}',
                           fontsize=11, fontweight='bold')

        axes[1].barh(range(20), top_sv[::-1], color=shap_colors[::-1])
        axes[1].set_yticks(range(20))
        axes[1].set_yticklabels(dim_labels[::-1], fontsize=7)
        axes[1].axvline(0, color='black', lw=0.8)
        axes[1].set_xlabel('SHAP value', fontsize=9)
        axes[1].set_title(f'Top-20 SHAP dims\n(T=text, I=image)', fontsize=11, fontweight='bold')

        fig.suptitle(f'SHAP Analysis — {SID_C} (Overall Satisfaction)',
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        shap_path = f'{xai_dir_C}/shap/{SID_C}_shap_analysis.png'
        os.makedirs(os.path.dirname(shap_path), exist_ok=True)
        fig.savefig(shap_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'[XAI] Đã lưu: {shap_path}')

        contrib_path = f'{xai_dir_C}/shap/{SID_C}_shap_contribution.json'
        with open(contrib_path, 'w') as f:
            json.dump(shap_contrib_C, f, indent=2)
    else:
        print('[SKIP] compute_shap_values thất bại.')
else:
    print('[SKIP] Không extract được fused embeddings.')

## 4.6 · LIME — Giải thích cục bộ (Text + Image)

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_C = {}
lime_text_exp_C     = None
lime_image_exp_C    = None
lime_img_paths_C    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_C = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_C, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_C = []
if lime_text_exp_C is not None:
    raw_weights_C = lime_text_exp_C.as_list(label=1)
    lime_text_weights_C = dict(raw_weights_C)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_C, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lt_path = f'{xai_dir_C}/lime/{SID_C}_lime_text_weights.json'
    os.makedirs(os.path.dirname(lt_path), exist_ok=True)
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_C, f, ensure_ascii=False, indent=2)
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_C['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_C = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_C, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_C is not None:
        lime_img_paths_C = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_C,
            original_image=sample_C['loaded_images'][0],
            save_dir=f'{xai_dir_C}/lime',
            sample_id=SID_C,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_C:
    w_sorted = sorted(lime_text_weights_C.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_C.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_C} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{xai_dir_C}/{SID_C}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')

## 4.7 · AI Agent — Báo cáo tổng hợp

> AI Agent (GPT-4o) phân tích evidence XAI và viết báo cáo tiếng Việt.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_C = check_sample_artifacts(SID_C, XAI_DIR)

add_cross_method_row(
    SID_C, pred_result_C,
    shap_contrib_C,
    raw_weights_C,
    case_type=CASE_C,
)

agent_output_C = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_C} ...')
        agent_C = ExplanationAgent(agent_config)
        agent_output_C = run_safe(
            agent_C.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_C,
            review_text=sample_C['text'],
            predictions=pred_result_C['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_C['ground_truth'],
            case_type=CASE_C,
            language='vi',
            mode='full',
            num_images=sample_C['num_real_images'],
            output_dir=agent_dir_C,
        )
        if agent_output_C:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_C.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print("  → os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_C, pred_result_C, artifact_check_C)
print(f'✅ HOÀN THÀNH MẪU C: {SID_C}')

---
# 📊 PHẦN 5 — SO SÁNH CROSS-SAMPLE

Tổng hợp kết quả 3 mẫu: MAE per target, đóng góp modality SHAP, top LIME words.

In [ ]:
if CROSS_METHOD_ROWS:
    df_cross = pd.DataFrame(CROSS_METHOD_ROWS)
    print('╔══ BẢNG SO SÁNH 3 MẪU ══════════════════════════════════╗')
    print(df_cross.to_string(index=False))
    print('╚═════════════════════════════════════════════════════════╝')
    display(df_cross)
else:
    print('[WARN] Chưa có dữ liệu cross-method.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

all_samples = [
    ('A', SAMPLE_IDS['A'], pred_result_A, shap_contrib_A),
    ('B', SAMPLE_IDS['B'], pred_result_B, shap_contrib_B),
    ('C', SAMPLE_IDS['C'], pred_result_C, shap_contrib_C),
]

# Panel 1: MAE per target
x = range(len(TARGET_NAMES))
width = 0.25
for i, (letter, sid, pred_res, _) in enumerate(all_samples):
    ae_vals = [pred_res['absolute_errors'][n] for n in TARGET_NAMES]
    offset  = (i - 1) * width
    axes[0].bar([xi + offset for xi in x], ae_vals, width,
                label=f'Mẫu {letter}', alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(['Food','Price','Atmos','Service','Overall'],
                         fontsize=8, rotation=20, ha='right')
axes[0].set_ylabel('Absolute Error', fontsize=9)
axes[0].set_title('MAE per Target\n(3 mẫu)', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=7)
axes[0].axhline(y=0.5, color='green', linestyle='--', alpha=0.5)

# Panel 2: SHAP modality contribution stacked bar
shap_t = [c.get('text_pct', 0) if c else 0 for _, _, _, c in all_samples]
shap_i = [c.get('image_pct', 0) if c else 0 for _, _, _, c in all_samples]
x2 = range(len(all_samples))
labels2 = [f'Mẫu {l}' for l, _, _, _ in all_samples]
axes[1].bar(x2, shap_t, label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'], alpha=0.8)
axes[1].bar(x2, shap_i, bottom=shap_t, label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'], alpha=0.8)
axes[1].set_xticks(list(x2))
axes[1].set_xticklabels(labels2, fontsize=9)
axes[1].set_ylabel('SHAP Contribution (%)', fontsize=9)
axes[1].set_title('SHAP Modality\nContribution', fontsize=10, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 100)

# Panel 3: Overall MAE
maes = [r['mean_mae'] for _, _, r, _ in all_samples]
cols_mae = ['#43A047', '#E53935', '#1E88E5']
bars3 = axes[2].bar(
    ['Mẫu A\n(Chính xác)', 'Mẫu B\n(Lỗi)', 'Mẫu C\n(Đa ảnh)'],
    maes, color=cols_mae, alpha=0.85, edgecolor='white')
for bar, mae_v in zip(bars3, maes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{mae_v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[2].set_ylabel('Mean MAE', fontsize=9)
axes[2].set_title('Overall MAE\n(3 mẫu)', fontsize=10, fontweight='bold')
axes[2].axhline(y=0.5, color='green', linestyle='--', alpha=0.6)

fig.suptitle('Cross-Sample Comparison — XAI Pipeline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
cross_path = f'{DEMO_ROOT}/cross_sample_comparison.png'
fig.savefig(cross_path, dpi=THESIS_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {cross_path}')

In [ ]:
# ── Tổng kết demo ────────────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════╗')
print('║            TỔNG KẾT DEMO XAI + AI AGENT                 ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Pipeline đầy đủ:                                        ║')
print('║    1. ✅ Dự đoán (CrossAttentionFusion)                  ║')
print('║    2. ✅ Grad-CAM (vùng ảnh quan trọng)                  ║')
print('║    3. ✅ PhoBERT Attention (từ nổi bật)                   ║')
print('║    4. ✅ Cross-Attention T2I + I2T                        ║')
print('║    5. ✅ SHAP (text/image modality contribution)          ║')
print('║    6. ✅ LIME (text + image, 4-panel)                     ║')
print('║    7. ✅ AI Agent (GPT-4o, báo cáo tiếng Việt)           ║')
print('║                                                          ║')
print('║  3 mẫu: Chính xác | Lỗi/Xung đột | Đa ảnh phong phú   ║')
print('╚══════════════════════════════════════════════════════════╝')

import glob as _glob
artifact_count = len(_glob.glob(f'{DEMO_ROOT}/**/*', recursive=True))
print(f'\nTổng files đã lưu vào Drive: {artifact_count}')